# Clinical Atlas: 3D Knee Reconstruction Pipeline

This notebook demonstrates the end-to-end 3D reconstruction of patient-specific knee anatomy from medical image segmentation masks (ground truth or model prediction).

### Pipeline Architecture
1. **Load Medical Image:** Read spacing, origin, direction matrix, affine (mm physical calibration)
2. **Extract Anatomical Labels:** Configurable label map (femur, tibia, cartilages)
3. **Connected Components:** Remove tiny disconnected speckles / keep primary bone structure
4. **Marching Cubes:** Extract high-resolution triangular surface mesh
5. **Physical Coordinate Conversion:** Voxel coordinates -> real physical space (mm)
6. **Taubin Smoothing:** Remove voxel staircase artifacts with minimal volume shrinkage
7. **Mesh Quality Assurance (QA):** Vertices, faces, volume, surface area, watertightness, volume drift %
8. **Export:** Save raw and smoothed `.stl` and `.obj` files + `mesh_manifest.json` + `mesh_qa.json`

In [ ]:
# CELL 1: Import Libraries
import os
import sys
import json
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Ensure reconstruction3d module is accessible from notebooks directory or workspace root
for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "reconstruction3d").is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from reconstruction3d import (
    generate_meshes,
    load_medical_image,
    get_image_metadata,
    get_label_voxel_counts,
    DEFAULT_LABEL_MAP
)

print("All libraries imported successfully!")
print(f"SimpleITK version: {sitk.__version__}")
print(f"Trimesh version:   {trimesh.__version__}")
print(f"Default Label Map: {DEFAULT_LABEL_MAP}")

In [ ]:
# CELL 2: Set Segmentation File Path
# Specify the path to your dataset ground-truth NIfTI file (or future model prediction)
# e.g. Path('data/OAIZIB_CM/case_001/segmentation.nii.gz')
SEGMENTATION_FILE = Path("data/sample_knee_seg.nii.gz")
OUTPUT_DIR = Path("outputs/case_001")
CASE_ID = "case_001"

# Self-contained fallback: If your dataset file is not yet at that path,
# this generates a synthetic knee demonstration volume so the notebook runs immediately!
if not SEGMENTATION_FILE.is_file():
    print(f"[NOTE] Specified file not found at: {SEGMENTATION_FILE}")
    print("[INFO] Creating a demonstration synthetic knee volume (Femur, Cartilage, Tibia)...")
    demo_dir = Path("outputs/demo_dataset")
    demo_dir.mkdir(parents=True, exist_ok=True)
    SEGMENTATION_FILE = demo_dir / "knee_ground_truth_demo.nii.gz"
    
    vol = np.zeros((80, 80, 80), dtype=np.uint8)
    z, y, x = np.ogrid[:80, :80, :80]
    # Label 1: Femur
    vol[np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.2)**2 + ((z-55)/1.4)**2) <= 18] = 1
    # Label 2: Femoral Cartilage
    vol[(np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.2)**2 + ((z-55)/1.4)**2) > 18) & (np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.2)**2 + ((z-55)/1.4)**2) <= 21) & (z <= 50)] = 2
    # Label 3: Tibia
    vol[np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.1)**2 + ((z-20)/1.3)**2) <= 16] = 3
    # Label 4: Medial Tibial Cartilage
    vol[(np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.1)**2 + ((z-20)/1.3)**2) > 16) & (np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.1)**2 + ((z-20)/1.3)**2) <= 19) & (z >= 24) & (x < 40)] = 4
    # Label 5: Lateral Tibial Cartilage
    vol[(np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.1)**2 + ((z-20)/1.3)**2) > 16) & (np.sqrt(((x-40)/1.1)**2 + ((y-40)/1.1)**2 + ((z-20)/1.3)**2) <= 19) & (z >= 24) & (x >= 40)] = 5
    
    demo_sitk = sitk.GetImageFromArray(vol)
    demo_sitk.SetSpacing((0.5, 0.5, 1.0))
    demo_sitk.SetOrigin((0.0, 0.0, 0.0))
    demo_sitk.SetDirection((1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0))
    sitk.WriteImage(demo_sitk, str(SEGMENTATION_FILE))
    print(f"[OK] Demo volume generated: {SEGMENTATION_FILE}")

print(f"Active Segmentation Path: {SEGMENTATION_FILE.resolve()}")
print(f"Output Directory:         {OUTPUT_DIR.resolve()}")

In [ ]:
# CELL 3: Display Image Metadata
image_data = load_medical_image(SEGMENTATION_FILE)
metadata = get_image_metadata(image_data)

print("=" * 60)
print("MEDICAL IMAGE METADATA & PHYSICAL CALIBRATION")
print("=" * 60)
print(f"Dimensions (Voxels X, Y, Z): {metadata['dimensions_voxels_xyz']}")
print(f"Array Shape (Z, Y, X):       {metadata['array_shape_zyx']}")
print(f"Voxel Spacing (mm X, Y, Z):  {metadata['voxel_spacing_mm_xyz']}")
print(f"Voxel Unit Volume:           {metadata['voxel_volume_mm3']} mm³")
print(f"Physical Origin (mm X, Y, Z):{metadata['physical_origin_mm_xyz']}")
print(f"Direction Cosine Matrix:\n{np.array(metadata['direction_matrix'])}")
print(f"Available Label IDs in Mask: {metadata['available_label_ids']}")
print("=" * 60)

In [ ]:
# CELL 4: Display Voxel Counts per Anatomical Structure
label_counts = get_label_voxel_counts(
    label_array=image_data["array"],
    spacing=image_data["spacing"],
    label_map=DEFAULT_LABEL_MAP
)

print(f"{'Label ID':<10} | {'Structure Name':<26} | {'Voxel Count':<14} | {'Volume (mm³)':<14} | {'Status'}")
print("-" * 80)
for lid, info in label_counts.items():
    status = "Present" if info["present"] else "ABSENT (Will Skip)"
    print(f"{lid:<10} | {info['name']:<26} | {info['voxel_count']:<14,d} | {info['volume_mm3']:<14,.2f} | {status}")

In [ ]:
# CELL 5: Run 3D Reconstruction Pipeline
results = generate_meshes(
    segmentation_path=SEGMENTATION_FILE,
    output_directory=OUTPUT_DIR,
    label_map=DEFAULT_LABEL_MAP,
    case_id=CASE_ID,
    taubin_lambda=0.5,
    taubin_nu=-0.53,
    taubin_iterations=20,
    keep_largest_component=True,
    export_stl=True,
    export_obj=True
)

print("\n" + "=" * 60)
print(f"Reconstruction Status: {results['status'].upper()}")
print(f"Total Anatomical Structures Reconstructed: {results['manifest']['total_structures']}")
print("=" * 60)

In [ ]:
# CELL 6: Print Generated Output Files
print("Generated Files on Disk:")
for file_path in results["generated_files"]:
    p = Path(file_path)
    size_kb = p.stat().st_size / 1024
    print(f"  [+] {p.name:<34} ({size_kb:8.1f} KB) -> {p.parent.name}/{p.name}")

print("\nMesh Manifest JSON preview:")
with open(OUTPUT_DIR / "mesh_manifest.json", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# CELL 7: Display Mesh QA Results
with open(OUTPUT_DIR / "mesh_qa.json", encoding="utf-8") as f:
    qa_data = json.load(f)

print("=" * 95)
print(f"MESH QUALITY ASSURANCE (QA) - CASE: {qa_data.get('case_id')}")
print("=" * 95)
header = f"{'Structure':<24} | {'Stage':<8} | {'Vertices':<10} | {'Faces':<10} | {'Volume (mm³)':<14} | {'Watertight':<10} | {'Drift (%)':<10}"
print(header)
print("-" * len(header))

for name, report in qa_data.items():
    if name in ["case_id", "metadata"]:
        continue
    raw = report["raw"]
    smooth = report["smoothed"]
    drift = report["volume_drift_percent"]
    print(f"{name:<24} | {'Raw':<8} | {raw['vertices']:<10,d} | {raw['faces']:<10,d} | {raw['volume_mm3']:<14,.2f} | {str(raw['watertight']):<10} | {'--':<10}")
    print(f"{' ':<24} | {'Smooth':<8} | {smooth['vertices']:<10,d} | {smooth['faces']:<10,d} | {smooth['volume_mm3']:<14,.2f} | {str(smooth['watertight']):<10} | {f'{drift:+.2f}%':<10}")
    print("-" * len(header))

In [ ]:
# CELL 8: Visualize One Generated Mesh (Femur) in Python
femur_path = OUTPUT_DIR / "meshes" / "femur_smooth.stl"

if femur_path.is_file():
    mesh = trimesh.load(str(femur_path))
    # Decimate slightly if high resolution for smooth interactive rendering
    display_mesh = mesh.simplify_quadric_decimation(10000) if len(mesh.faces) > 10000 else mesh
    
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")
    poly = Poly3DCollection(
        display_mesh.vertices[display_mesh.faces],
        alpha=0.85,
        edgecolor="#333333",
        linewidths=0.1
    )
    poly.set_facecolor("#e0d6c3") # Bone ivory tone
    ax.add_collection3d(poly)
    
    bounds = display_mesh.bounding_box.bounds
    ax.set_xlim(bounds[0][0], bounds[1][0])
    ax.set_ylim(bounds[0][1], bounds[1][1])
    ax.set_zlim(bounds[0][2], bounds[1][2])
    ax.set_xlabel("X (mm)")
    ax.set_ylabel("Y (mm)")
    ax.set_zlabel("Z (mm)")
    ax.set_title("Reconstructed Femur Mesh (Physical mm Coordinates)", fontsize=13, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.show()
else:
    print(f"Mesh file not found at {femur_path}")

In [ ]:
# CELL 9: Visualize All Available Structures Together (Anatomical Assembly)
color_palette = {
    "femur": ("#d9caa8", 0.75),                    # Distal femur bone
    "femoral_cartilage": ("#00acc1", 0.9),         # Femoral cartilage cap (cyan)
    "tibia": ("#b8afa0", 0.75),                    # Proximal tibia bone
    "medial_tibial_cartilage": ("#43a047", 0.9),   # Medial meniscus/cartilage (green)
    "lateral_tibial_cartilage": ("#fb8c00", 0.9),  # Lateral meniscus/cartilage (orange)
}

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
all_bounds = []

for structure_name, (color, alpha) in color_palette.items():
    stl_path = OUTPUT_DIR / "meshes" / f"{structure_name}_smooth.stl"
    if not stl_path.is_file():
        continue
    m = trimesh.load(str(stl_path))
    m_disp = m.simplify_quadric_decimation(6000) if len(m.faces) > 6000 else m
    poly = Poly3DCollection(m_disp.vertices[m_disp.faces], alpha=alpha, edgecolor="none")
    poly.set_facecolor(color)
    ax.add_collection3d(poly)
    all_bounds.append(m.bounding_box.bounds)

if all_bounds:
    bounds_arr = np.array(all_bounds)
    min_pt = np.min(bounds_arr[:, 0, :], axis=0)
    max_pt = np.max(bounds_arr[:, 1, :], axis=0)
    ax.set_xlim(min_pt[0], max_pt[0])
    ax.set_ylim(min_pt[1], max_pt[1])
    ax.set_zlim(min_pt[2], max_pt[2])

ax.set_xlabel("X (mm, Medial/Lateral)")
ax.set_ylabel("Y (mm, Anterior/Posterior)")
ax.set_zlabel("Z (mm, Inferior/Superior)")
ax.set_title("Multi-Structure Knee Joint Assembly (Physical mm)", fontsize=14, fontweight="bold", pad=20)
ax.view_init(elev=15, azim=45)
plt.tight_layout()
plt.show()